TDE1 - Análise de Revisões de Compras
Dataset: Flipkart Laptop Reviews — Kaggle

Discentes: Maria Taliciane Pereira da Silva, Wesley Furtado Pessoa, Michel Gomes Pinheiro, Ingrid Sampaio Angelim do Nascimento, Moryak Samyak Arrais Benicio, José Daniel Macêdo Fechine, Clara Sobreira Vidal e Lia Costa Andrade

Professor: Alexandro Oliveira Alexandrino

Disciplina: Inteligência Artificial

Data: 30/08/2026

In [ ]:
import os
import re
import glob
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from scipy import stats

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 80)

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelsize"] = 11

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Bibliotecas importadas e ambiente configurado com sucesso.")

In [ ]:
import kagglehub

path = kagglehub.dataset_download("gitadityamaddali/flipkart-laptop-reviews")

arquivos_csv = glob.glob(os.path.join(path, "*.csv"))
assert arquivos_csv, f"Nenhum arquivo CSV encontrado em: {path}"
df = pd.read_csv(arquivos_csv[0])

print("Path to dataset files:", path)
print("DataFrame carregado:", os.path.basename(arquivos_csv[0]), df.shape)

100%|██████████| 0.98M/0.98M [00:00<00:00, 72.1MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/gitadityamaddali/flipkart-laptop-reviews/versions/1
DataFrame carregado: laptops_dataset_final_600.csv (24113, 7)


In [ ]:
# Garantia de que o DataFrame principal foi criado corretamente antes de qualquer análise
assert "df" in globals(), "O DataFrame 'df' não foi criado. Verifique a célula de carregamento acima."
assert df.shape[0] > 0, "O DataFrame está vazio."
df.head(3)

,product_name,overall_rating,no_ratings,no_reviews,rating,title,review
0,Apple MacBook AIR Apple M2 - (8 GB/256 GB SSD/Mac OS Monterey)...,4.7,"15,210",900,5,Perfect product!,"Loved it, it's my first MacBook that I earned from my hardwork 🥺❤️"
1,Apple MacBook AIR Apple M2 - (8 GB/256 GB SSD/Mac OS Monterey)...,4.7,"15,210",900,5,Fabulous!,Battery lasted longer than my first relationship (2 days).\nListening to Ari...
2,Apple MacBook AIR Apple M2 - (8 GB/256 GB SSD/Mac OS Monterey)...,4.7,"15,210",900,5,Fabulous!,Such a great deal.. very happy with the performance and battery life..Origio...


## 1. Contextualização e Enquadramento do Estudo


A **Flipkart**, um dos pilares do comércio eletrônico indiano, espelha a dinâmica global de grandes varejistas online, onde milhões de consumidores compartilham suas percepções sobre produtos adquiridos. Neste contexto, o foco recai sobre as avaliações de **notebooks/laptops**. Cada avaliação constitui um compêndio de informações: uma **classificação estelar de 1 a 5**, um **título conciso** (e.g., 'Excelente', 'Satisfatório', 'Apenas razoável') e um **discurso textual livre** que narra a experiência do usuário.

O cerne deste projeto reside na seguinte interrogação: **seria possível inferir a nota (rating) atribuída por um cliente com base exclusivamente no conteúdo textual de sua avaliação?** Esta questão possui relevância prática considerável, uma vez que organizações frequentemente se deparam com volumes massivos de texto não estruturado (avaliações, tíquetes de suporte, comentários em mídias sociais) nem sempre acompanhados de uma avaliação numérica explícita. Adicionalmente, a nota pode estar ausente, ser imprecisa ou desatualizada. A capacidade de um classificador de sentimento ou de nota permite o **monitoramento escalável da satisfação do cliente**, transcendendo a dependência da inserção manual de classificações.

### Propósito da Análise Atual

Esta fase inicial do TDE1, tem como desígnio primordial a **compreensão aprofundada dos dados antes de qualquer incursão em modelagem**. Almeja-se desvendar a estrutura, a integridade, a distribuição e as inter-relações entre as variáveis. As descobertas aqui delineadas servirão como alicerce para decisões estratégicas futuras, tais como a seleção de técnicas de balanceamento de classes, a identificação de variáveis numéricas passíveis de atuação como `features` auxiliares, a abordagem para o tratamento de duplicatas e `outliers`, e a antecipação da complexidade inerente à discriminação entre as classes de `rating` a partir da análise textual.

### Gênese dos Dados

Os dados em questão foram diligentemente coletados por meio de *web scraping* diretamente da plataforma Flipkart e subsequentemente disponibilizados publicamente no Kaggle, sob o título
`'Laptop reviews dataset (flipkart)'` ([https://www.kaggle.com/datasets/gitadityamaddali/flipkart-laptop-reviews](https://www.kaggle.com/datasets/gitadityamaddali/flipkart-laptop-reviews)). Este conjunto compreende **24 mil avaliações** feitas por clientes reais acerca de **centenas de modelos de notebooks** comercializados no referido `marketplace`.

### Interpretação da Unidade de Observação

Cada registro (linha) no conjunto de dados representa uma **avaliação singular de um cliente sobre um produto (laptop) específico**. Um mesmo produto pode, naturalmente, figurar em múltiplas linhas, correspondendo a cada cliente que o avaliou.

### Variáveis Primordiais do Dataset

| Coluna | Significado Acadêmico | Implicações Humanizadas |
|---|---|---|
| `product_name` | Identificação nominal completa do notebook, incluindo especificações técnicas (marca, processador, RAM, armazenamento, etc.). | Revela a especificidade do item avaliado, fundamental para o contexto da experiência do usuário. |
| `overall_rating` | Média ponderada das avaliações do produto, consolidada até o momento da coleta de dados. | Representa a percepção coletiva sobre a qualidade do produto, oferecendo um panorama da satisfação geral dos consumidores. |
| `no_ratings` | Quantidade total de classificações numéricas (estrelas) recebidas pelo produto até a data da coleta. | Indica a popularidade e a amplitude de feedback recebido pelo produto, refletindo a sua visibilidade no mercado. |
| `no_reviews` | Volume total de avaliações textuais (comentários escritos) associadas ao produto. | Quantifica a disposição dos consumidores em detalhar suas experiências, um recurso valioso para análise qualitativa. |
| `rating` | Classificação individual de **1 a 5 estrelas** atribuída pelo cliente em uma avaliação específica — **variável-alvo** do projeto. | Traduz o veredicto pessoal do cliente, sua percepção imediata e subjetiva sobre o produto, sendo o foco de nossa predição. |
| `title` | Título conciso da avaliação, formulado pelo cliente. | Uma síntese da impressão inicial ou do ponto principal da crítica, oferecendo um vislumbre rápido do sentimento geral. |
| `review` | Texto livre e detalhado da avaliação escrita pelo cliente. | O coração da experiência do consumidor, onde são expressas as nuances, os pormenores e as emoções que fundamentam a nota atribuída. |

### A Conexão com o Desafio de Classificação Multiclasse

A variável `rating` pode assumir um de cinco valores discretos (1, 2, 3, 4 e 5), o que configura intrinsecamente um problema de **classificação multiclasse**. O objetivo prospectivo é desenvolver um modelo capaz de predizer corretamente uma dessas cinco classes, utilizando como base o conteúdo textual da `review` (e, possivelmente, o `title`). Conforme será pormenorizado na seção de Análise Univariada, a distribuição dessas classes exibe um **desbalanceamento inerente**. Tal característica justifica a adoção da **Acurácia Balanceada** como métrica de avaliação, a qual atribui igual ponderação a cada classe, prevenindo que o desempenho global seja distorcido pela classe majoritária. Ademais, este achado sugere a imperatividade de estratégias de balanceamento de dados nas etapas subsequentes do projeto.

### A Imperatividade da Análise de Avaliações

Decifrar as particularidades que distinguem uma avaliação de nota 1 de uma de nota 5 - seja em termos de extensão textual, vocabulário empregado ou padrões de escrita -, representa o passo inaugural e fundamental para a construção de um classificador fidedigno. Adicionalmente, o discernimento de eventuais questões de integridade dos dados (tais como duplicatas, textos excessivamente concisos ou desbalanceamento acentuado de classes) é crucial para evitar inferências equívocas nas fases subsequentes de Processamento de Linguagem Natural (NLP) e modelagem. Uma análise meticulosa, portanto, não só pavimenta o caminho para a robustez do modelo, mas também salvaguarda a validade das conclusões derivadas.